# 08a — EDA phần A: Ở ĐÂU (người phụ trách: Lucknow/Jaipur)

**Câu hỏi nghiên cứu chung**

> **Ô nhiễm không khí nặng (AQI từ 201 trở lên) tập trung ở thành phố nào và vào mùa nào trong 8 thành phố Ấn Độ giai đoạn 2015–2020, để cơ quan môi trường và người dân biết nên ưu tiên cảnh báo sức khỏe và giảm phát thải ở đâu, lúc nào?**


**Quy ước chung** (chi tiết ở `08_eda.ipynb`): so sánh giữa thành phố chỉ dùng `win` (khoảng chung); "nặng" = AQI >= 201 trên số ngày **có AQI**;
mùa: Đông (12-2), Hè (3-5), Mưa (6-9), Sau mưa (10-11); chỉ viết "có liên quan", **không** viết "gây ra".
Hàm dùng chung nằm ở `src/eda_utils.py` (không sửa nếu chưa báo nhóm).

**Làm việc nhóm:** chỉ sửa notebook **này** và chỉ lưu hình có tên của phần mình (`eda_<số>_...png`). `git pull` trước khi push.

## Việc cần làm
1. Bảng thống kê 4 họ theo thành phố (hàm đã có, hãy đọc và giải thích từng cột bằng lời).
2. **Biểu đồ 1**: % ngày AQI >= 201 theo thành phố (cột ngang, sắp giảm dần, từ 0, tô nhấn thành phố cao nhất).
3. **Biểu đồ 5**: phân phối AQI theo thành phố (boxplot, sắp theo trung vị) để thấy phân tán và hình dạng.

## Lưu ý đã phát hiện khi kiểm tra
- Các thành phố có tỉ lệ gần nhau **đổi thứ hạng khi bỏ ngẫu nhiên 10% dòng** (xem bảng kiểm tra độ nhạy ở `08_eda.ipynb`): Lucknow và Gurugram, Amritsar và Jaipur, Bengaluru và Hyderabad. Vì vậy hãy **nhóm các thành phố** (ví dụ nhóm rất nặng, nhóm trung bình, nhóm gần như không) thay vì xếp hạng từng thành phố.
- Bảng `skew` > 0 nghĩa là phân phối lệch phải (nhiều ngày thấp, ít ngày rất cao); giải thích điều này bằng lời vì nó quyết định nên dùng **trung vị** hay trung bình.

**Câu vấn đáp phải trả lời được**: confounder nào có thể giải thích khác biệt giữa các thành phố? (gợi ý: số và vị trí trạm đo, quy mô đô thị, địa lý, thời tiết; dữ liệu này **không** có thời tiết và không có vị trí trạm.) Biểu đồ này có thể đánh lừa người đọc vội ở chỗ nào?

## Chuẩn biểu đồ (W7), tick đủ trước khi push

- [ ] Tiêu đề là **kết luận có con số** (không phải "AQI theo thành phố")
- [ ] Trục có **nhãn và đơn vị**
- [ ] Biểu đồ cột **bắt đầu từ 0**, sắp theo giá trị (không theo bảng chữ cái)
- [ ] **Một** màu nhấn cho điều muốn người đọc thấy, còn lại màu trung tính
- [ ] Ghi nhãn trực tiếp, bỏ legend nếu chỉ có một nhóm
- [ ] Không dùng biểu đồ tròn, 3D, trục kép; không dùng chỉ đỏ-xanh lá để phân biệt
- [ ] Thử chuyển sang thang xám vẫn đọc được
- [ ] Chú thích có cỡ mẫu và ngày dữ liệu (hàm `finish_chart` đã làm sẵn)

## 0. Setup

In [1]:
import sys, os
sys.path.append(os.path.abspath("../src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from eda_utils import (CITIES, USABLE, SEASON_ORDER, ACCENT, NEUTRAL,
                       vn, load, poor_share, four_families, finish_chart, save_fig, set_style)

set_style()
eda, win, START, END = load()      # eda: toàn bộ 2015-2020 | win: khoảng chung của 8 thành phố (dùng khi so sánh giữa thành phố)

Toàn bộ: 14.056 dòng, 01/01/2015 - 01/07/2020
Khoảng chung (biến win): 14/06/2017 - 01/07/2020, 8.912 dòng; 8.734 ngày có AQI


## A1. Bảng 4 họ thống kê (AQI, khoảng chung)

In [2]:
stats_city = four_families(win, "AQI", "City").sort_values("median", ascending=False)
stats_city

,n,mean,median,std,iqr,skew,spearman_PM2.5
City,,,,,,,
Delhi,1105,225.38,205.0,121.09,192.00,0.66,0.96
Lucknow,1111,206.98,188.0,112.69,211.00,0.43,0.96
Gurugram,1070,211.55,187.5,112.45,182.00,0.79,0.94
Jaipur,1093,133.52,122.0,54.09,63.00,1.33,0.88
Amritsar,1029,122.45,104.0,76.84,64.00,2.94,0.81
Hyderabad,1108,92.56,89.0,35.22,56.25,0.38,0.93
Chennai,1110,98.94,87.0,39.71,37.00,2.19,0.69
Bengaluru,1108,86.58,81.0,27.57,37.00,1.65,0.80


## A2. Biểu đồ 1: % ngày AQI >= 201 theo thành phố

In [3]:
share_city = poor_share(win, "City").sort_values("pct_poor")     # tăng dần để cột lớn nằm trên cùng
share_city
# TODO (người A): vẽ cột ngang theo khuôn của biểu đồ mẫu (xem 08_eda.ipynb), tiêu đề nêu kết luận có số,
# nhấn thành phố cao nhất, lưu bằng save_fig(fig, "01_poor_share_city")

,pct_poor,n_days
City,,
Hyderabad,0.1,1108
Bengaluru,0.2,1108
Chennai,3.1,1110
Jaipur,11.1,1093
Amritsar,11.4,1029
Gurugram,46.3,1070
Lucknow,46.4,1111
Delhi,51.0,1105


## A3. Biểu đồ 5: phân phối AQI theo thành phố

In [4]:
# TODO (người A): boxplot AQI theo thành phố (sắp theo trung vị), tô nhấn một thành phố,
# lưu bằng save_fig(fig, "05_aqi_distribution")

## Phát hiện của phần A (người ghép báo cáo sẽ gom bảng này vào mục 5)

Mỗi phát hiện phải gắn **một biểu đồ** (không có biểu đồ thì chỉ là ý kiến), kèm confounder và hạn chế.

| # | Phát hiện (có con số) | Biểu đồ | Confounder có thể | Hạn chế |
|---|---|---|---|---|
| A1 | | 1 | | |
| A2 | | 5 | | |

**Trước khi push:** Restart & Run All không lỗi; đã tick checklist biểu đồ; đã `git pull`.